In [1]:
from pathlib import Path
import pandas as pd

DATA = Path("../data/scifact")

corpus = pd.read_json(DATA / "corpus.jsonl", lines=True, dtype={"_id": str})
queries = pd.read_json(DATA / "queries.jsonl", lines=True, dtype={"_id": str})
qrels = pd.read_csv(DATA / "qrels" / "test.tsv", sep="\t", dtype=str)

print(corpus.shape, queries.shape, qrels.shape)
corpus.head(3)

(5183, 4) (1109, 3) (339, 3)


,_id,title,text,metadata
0,4983,Microstructural development of human newborn c...,Alterations of the architecture of cerebral wh...,{}
1,5836,Induction of myelodysplasia by myeloid-derived...,Myelodysplastic syndromes (MDS) are age-depend...,{}
2,7912,"BC1 RNA, the transcript from a master gene for...",ID elements are short interspersed elements (S...,{}


In [2]:
# want to know the number of uique queries in the queries dataframe to be able to see how many we can use for evaluating
print(qrels.columns.tolist())
print(qrels.head())
print("unique test queries:", qrels.iloc[:, 0].nunique())
print("score values:", qrels.iloc[:, 2].value_counts().to_dict())
print("labels per query:")
print(qrels.groupby(qrels.columns[0]).size().value_counts().sort_index())

['query-id', 'corpus-id', 'score']
  query-id corpus-id score
0        1  31715818     1
1        3  14717500     1
2        5  13734012     1
3       13   1606628     1
4       36   5152028     1
unique test queries: 300
score values: {'1': 339}
labels per query:
1    277
2     14
3      4
4      3
5      2
Name: count, dtype: int64


In [3]:
# look at unique ids and check the empty titles and texts as an empty q/a cant be matched to anything else and then we check the word limit so that we can see if our model limits can handle the larger bits of text. 
print("ids unique:", corpus["_id"].is_unique)
print("empty titles:", (corpus["title"].str.strip() == "").sum())
print("empty texts:", (corpus["text"].str.strip() == "").sum())

words = corpus["text"].str.split().str.len()
print(words.describe())

ids unique: True
empty titles: 0
empty texts: 0
count    5183.000000
mean      201.810920
std        86.007594
min        26.000000
25%       147.000000
50%       192.000000
75%       247.000000
max      1524.000000
Name: text, dtype: float64


In [4]:
test_ids = set(qrels["query-id"])
test_queries = queries[queries["_id"].isin(test_ids)]
print(len(test_queries), "queries usable for test")
print("test ids missing from queries file:", len(test_ids - set(queries["_id"])))

300 queries usable for test
test ids missing from queries file: 0


In [8]:
q = test_queries.iloc[1]
doc_id = qrels.loc[qrels["query-id"] == q["_id"], "corpus-id"].iloc[0]
doc = corpus[corpus["_id"] == doc_id].iloc[0]

print("QUERY:", q["text"])
print()
print("TITLE:", doc["title"])
print("ABSTRACT:", doc["text"])

QUERY: 1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.

TITLE: Rare Variants Create Synthetic Genome-Wide Associations
ABSTRACT: Genome-wide association studies (GWAS) have now identified at least 2,000 common variants that appear associated with common diseases or related traits (http://www.genome.gov/gwastudies), hundreds of which have been convincingly replicated. It is generally thought that the associated markers reflect the effect of a nearby common (minor allele frequency >0.05) causal site, which is associated with the marker, leading to extensive resequencing efforts to find causal sites. We propose as an alternative explanation that variants much less common than the associated one may create "synthetic associations" by occurring, stochastically, more often in association with one of the alleles at the common site versus the other allele. Although synthetic associations are an